In [0]:
# ============================================
# Gold Metadata (Temporary)
# ============================================

gold_tables = {

    "dim_customers": {
        "target_table": "gold.dim_customers",
        "merge_key": "CustomerSK"
    },

    "dim_branches": {
        "target_table": "gold.dim_branches",
        "merge_key": "BranchSK"
    },

    "dim_products": {
        "target_table": "gold.dim_products",
        "merge_key": "ProductSK"
    },

    "fact_sales": {
        "target_table": "gold.fact_sales",
        "merge_key": "OrderItemID"
    }

}

In [0]:
# ============================================================
# Configuration
# ============================================================

from pyspark.sql.functions import col

# Storage account
storage_account_name = "southbeanadlsdev"

# Gold Container
gold_base = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/"

# Silver Tables
silver_customers = "silver.customers"
silver_branches = "silver.branches"
silver_products = "silver.products"
silver_categories = "silver.categories"
silver_orders = "silver.orders"
silver_orderitems = "silver.orderitems"

# Gold Tables
gold_dim_customers = "gold.dim_customers"
gold_dim_branches = "gold.dim_branches"
gold_dim_products = "gold.dim_products"
gold_fact_sales = "gold.fact_sales"

# Gold Paths
dim_customers_path = gold_base + "dim_customers"
dim_branches_path = gold_base + "dim_branches"
dim_products_path = gold_base + "dim_products"
fact_sales_path = gold_base + "fact_sales"

print("Gold notebook configuration loaded.")

In [0]:
# ============================================================
# Build Gold Tables
# ============================================================

for table_name, config in gold_tables.items():

    print("=" * 60)
    print(f"Processing {table_name}")
    print("=" * 60)

    merge_key = config["merge_key"]
    target_table = config["target_table"]

    if table_name == "dim_customers":

        # ----------------------------------------
        # Build DimCustomer
        # ----------------------------------------

        # Read Silver
        customers_df = (
            spark.table(silver_customers)
                 .filter(col("IsCurrent") == True)
                 .select(
                     "CustomerSK",
                     "CustomerID",
                     "CustomerName",
                     "Phone",
                     "City",
                     "IsDeleted"
                 )
        )

        publish_to_gold(
            source_df=customers_df,
            target_table=target_table,
            merge_key=merge_key
        )

        print("✓ dim_customers completed.")


    elif table_name == "dim_branches":

        # ----------------------------------------
        # Build DimBranch
        # ----------------------------------------

        branches_df = (
            spark.table(silver_branches)
                 .filter(col("IsCurrent") == True)
                 .select(
                     "BranchSK",
                     "BranchID",
                     "BranchName",
                     "City",
                     "Manager",
                     "IsDeleted"
                 )
        )

        publish_to_gold(
            source_df=branches_df,
            target_table=target_table,
            merge_key=merge_key
        )

        print("✓ dim_branches completed.")


    elif table_name == "dim_products":

        # ----------------------------------------
        # Build DimProduct
        # ----------------------------------------

        products_df = (
            spark.table(silver_products)
                 .filter(col("IsCurrent") == True)
        )

        categories_df = spark.table(silver_categories)

        dim_products_df = (
            products_df.alias("p")
                .join(
                    categories_df.alias("c"),
                    "CategoryID",
                    "left"
                )
                .select(
                    "ProductSK",
                    "ProductID",
                    "ProductName",
                    "CategoryID",
                    "CategoryName",
                    "Price",
                    "IsDeleted"
                )
        )

        publish_to_gold(
            source_df=dim_products_df,
            target_table=target_table,
            merge_key=merge_key
        )

        print("✓ dim_products completed.")


    elif table_name == "fact_sales":

        # ----------------------------------------
        # Build FactSales
        # ----------------------------------------

        orders_df = spark.table(silver_orders)

        orderitems_df = spark.table(silver_orderitems)

        dim_customer_df = spark.table(gold_dim_customers)

        dim_branch_df = spark.table(gold_dim_branches)

        dim_product_df = spark.table(gold_dim_products)

        fact_sales_df = (
            orderitems_df.alias("oi")

            .join(
                orders_df.alias("o"),
                "OrderID"
            )

            .join(
                dim_customer_df.alias("dc"),
                col("o.CustomerID") == col("dc.CustomerID"),
                "left"
            )

            .join(
                dim_branch_df.alias("db"),
                col("o.BranchID") == col("db.BranchID"),
                "left"
            )

            .join(
                dim_product_df.alias("dp"),
                col("oi.ProductID") == col("dp.ProductID"),
                "left"
            )

            .select(
                col("oi.OrderItemID"),

                col("o.OrderID"),
                col("o.OrderDate"),

                col("dc.CustomerSK"),
                col("db.BranchSK"),
                col("dp.ProductSK"),

                col("oi.Quantity"),
                col("oi.UnitPrice"),
                col("oi.LineAmount"),

                col("o.TotalAmount")
            )
        )

        publish_to_gold(
            source_df=fact_sales_df,
            target_table=target_table,
            merge_key=merge_key
        )

        print("✓ fact_sales completed.")


print("\n✓ Gold layer completed successfully.")

In [0]:
from delta.tables import DeltaTable


def publish_to_gold(
    source_df,
    target_table,
    merge_key
):
    """
    Publishes data to Gold using Delta MERGE.

    Strategy:
        - Insert only
        - Atomic
        - Idempotent

    Existing records are ignored.
    New records are inserted.
    """

    target = DeltaTable.forName(spark, target_table)

    (
        target.alias("target")
        .merge(
            source_df.alias("source"),
            f"""
            target.{merge_key} = source.{merge_key}
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"✓ Published to {target_table}")